# LeetCode 416 — Partition Equal Subset Sum bằng Python

Link đề: https://leetcode.com/problems/partition-equal-subset-sum/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Knapsack (0-1)

Skeleton LeetCode:

```python
class Solution:
    def canPartition(self, nums: List[int]) -> bool:
        ...
```

Ghi chú: phần phát biểu đề được dịch sang tiếng Việt từ đề gốc; mọi đáp án trong notebook được chạy lại và đối chiếu với một oracle vét cạn độc lập (thử mọi tập con bằng bitmask).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho mảng số nguyên dương `nums`, hỏi có thể chia thành **hai tập con có tổng bằng nhau** hay không (mỗi phần tử thuộc đúng một tập).
2. Nắm công cụ Python: `sum`, `divmod`, `%` kiểm tra chẵn lẻ, `range` đi **ngược**, danh sách 2 chiều `[[False] * (m + 1) for _ in range(n + 1)]`, `functools.lru_cache`, số nguyên lớn làm **bitset** (`<<`, `>>`, `|`, `&`), `itertools`/bitmask.
3. Đi từ cách **thử mọi tập con `O(2^n)`** → **đệ quy có nhớ (top-down)** → **bảng DP 2 chiều** → **DP 1 chiều (duyệt ngược)** → **bitset bằng số nguyên Python**.
4. Hiểu mấu chốt: hai nửa bằng nhau ⇔ tồn tại tập con có tổng bằng **một nửa tổng**. Bài toán trở thành **0-1 knapsack** "có chọn được tập con tổng đúng `m` không".
5. Tránh các lỗi kinh điển: quên kiểm tra tổng lẻ, duyệt DP 1 chiều **xuôi** (dùng một phần tử nhiều lần), khởi tạo `dp[0]` sai, cắt tỉa sai khi có phần tử lớn hơn `m`, dùng `m = sum // 2` nhưng không loại tổng lẻ.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Trả về `true` nếu có thể **chia mảng thành hai tập con** sao cho **tổng các phần tử hai tập bằng nhau**, ngược lại trả về `false`.

**Ví dụ 1:** `nums = [1,5,11,5]` → `true`. Chia thành `[1, 5, 5]` và `[11]`, cùng tổng `11`.

**Ví dụ 2:** `nums = [1,2,3,5]` → `false`. Tổng `11` lẻ nên không thể chia đều.

**Ràng buộc:**

- `1 <= nums.length <= 200`
- `1 <= nums[i] <= 100`

In [1]:
from functools import lru_cache
import random
import time
from typing import List

examples = [
    ([1, 5, 11, 5], True),
    ([1, 2, 3, 5], False),
]
for nums, expected in examples:
    print(nums, "tổng =", sum(nums), "->", expected)

[1, 5, 11, 5] tổng = 22 -> True
[1, 2, 3, 5] tổng = 11 -> False


### 2.1. Đọc kỹ đề

- "Hai tập con" ở đây là chia **toàn bộ** mảng thành hai phần rời nhau (mỗi phần tử thuộc đúng một phần). Các phần tử trùng giá trị vẫn là **các phần tử khác nhau** (hai số `5` ở ví dụ 1 là hai phần tử).
- Mọi `nums[i] >= 1` (dương), nên tổng luôn tăng khi thêm phần tử — một tính chất giúp bitset và cắt tỉa.
- Tổng tối đa `200 * 100 = 20000`, nửa tổng tối đa `10000`. Bảng DP `n * m` tối đa khoảng `200 * 10000 = 2 * 10^6` ô: vừa phải.
- Không cần chỉ ra cách chia, chỉ cần `True/False`.

## 3. Kiến thức nền

### 3.1. Tổng, chẵn lẻ và `divmod`

`sum(nums)` cộng mọi phần tử. Tổng **lẻ** thì không thể chia thành hai nửa nguyên bằng nhau. `divmod(s, 2)` trả `(s // 2, s % 2)`: thương là nửa tổng, dư cho biết chẵn (`0`) hay lẻ (`1`).

In [2]:
nums = [1, 5, 11, 5]
s = sum(nums)
half, rem = divmod(s, 2)
print(s, half, rem)                  # 22 11 0
print(divmod(sum([1, 2, 3, 5]), 2))  # (5, 1): tổng 11 lẻ

22 11 0
(5, 1)


### 3.2. Danh sách 2 chiều (bảng) và cạm bẫy `[[...]] * n`

Tạo bảng `n + 1` hàng, `m + 1` cột toàn `False`: `[[False] * (m + 1) for _ in range(n + 1)]`. **Không** viết `[[False] * (m + 1)] * (n + 1)` vì như vậy mọi hàng là **cùng một đối tượng**, sửa một hàng thì tất cả đổi theo.

In [3]:
bad = [[0] * 3] * 2
bad[0][0] = 9
print("sai:", bad)                    # [[9, 0, 0], [9, 0, 0]]
good = [[0] * 3 for _ in range(2)]
good[0][0] = 9
print("đúng:", good)                  # [[9, 0, 0], [0, 0, 0]]

sai: [[9, 0, 0], [9, 0, 0]]
đúng: [[9, 0, 0], [0, 0, 0]]


### 3.3. `range` đi ngược

`range(start, stop, step)` với `step` âm đi lùi: `range(5, -1, -1)` cho `5, 4, 3, 2, 1, 0`. Ở DP 1 chiều ta duyệt `j` từ `m` xuống `x` bằng `range(m, x - 1, -1)`.

In [4]:
print(list(range(5, -1, -1)), list(range(10, 5 - 1, -1)))

[5, 4, 3, 2, 1, 0] [10, 9, 8, 7, 6, 5]


### 3.4. `functools.lru_cache`

`@lru_cache(maxsize=None)` ghi nhớ kết quả của hàm theo tham số: gọi lại với cùng tham số thì trả kết quả đã lưu, không tính lại. Nó biến đệ quy có nhiều bài toán con trùng lặp thành **DP top-down (memoization)**.

In [5]:
calls = 0


@lru_cache(maxsize=None)
def fib(n):
    global calls
    calls += 1
    return n if n < 2 else fib(n - 1) + fib(n - 2)


print(fib(30), "số lần thân hàm thực sự chạy:", calls)    # 31 thay vì hơn một triệu

832040 số lần thân hàm thực sự chạy: 31


### 3.5. Số nguyên Python làm bitset

Một số nguyên Python (không giới hạn độ lớn) có thể xem như một **dãy bit**: bit thứ `j` bật nghĩa là "tổng `j` đạt được".

- `1 << x`: số có đúng bit `x` bật.
- `bits << x`: dịch tất cả bit lên `x` vị trí — tức "cộng `x` vào mọi tổng đang đạt".
- `bits | other`: hợp hai tập.
- `(bits >> j) & 1`: kiểm tra bit `j`.

In [6]:
bits = 1                              # chỉ có tổng 0
for x in [1, 5, 11, 5]:
    bits |= bits << x                 # thêm khả năng "chọn x"
print(bin(bits))
print([j for j in range(23) if (bits >> j) & 1])    # các tổng đạt được
print("đạt được 11?", (bits >> 11) & 1 == 1)

0b11000110001110001100011
[0, 1, 5, 6, 10, 11, 12, 16, 17, 21, 22]
đạt được 11? True


### 3.6. Bitmask duyệt mọi tập con

Với `n` phần tử, số `mask` từ `0` đến `2^n - 1` mã hóa một tập con: bit `i` bật ⇔ phần tử `i` được chọn. `(mask >> i) & 1` kiểm tra bit `i`.

In [7]:
nums = [1, 5, 11, 5]
n = len(nums)
for mask in range(4):
    chosen = [nums[i] for i in range(n) if (mask >> i) & 1]
    print(f"mask={mask:04b} -> {chosen} tổng {sum(chosen)}")

mask=0000 -> [] tổng 0
mask=0001 -> [1] tổng 1
mask=0010 -> [5] tổng 5
mask=0011 -> [1, 5] tổng 6


## 4. Ý tưởng nền tảng: tính tay

`nums = [1, 5, 11, 5]`, tổng `22` chẵn, cần tìm tập con tổng `11`.

Duyệt từng số và liệt kê **mọi tổng đạt được** (ban đầu chỉ `{0}`):

- Thêm `1`: `{0, 1}`.
- Thêm `5`: `{0, 1, 5, 6}`.
- Thêm `11`: `{0, 1, 5, 6, 11, 12, 16, 17}` ⇒ đã có `11` ✓.
- Thêm `5` nữa: `{0,1,5,6,10,11,12,16,17,21,22,...}`.

Vì `11` xuất hiện ⇒ chia được (`[11]` và phần còn lại `1+5+5 = 11`).

Ví dụ 2 `[1,2,3,5]` tổng `11` lẻ ⇒ không cần tính, `false`. Nếu ta thử `[1,2,5]` (tổng `8`, cần `4`): tổng đạt được `{0,1,2,3,5,6,7,8}` — không có `4` ⇒ `false`.

**Quy trình:** (1) tổng lẻ ⇒ `false`. (2) `m = tổng / 2`. (3) tính tập các tổng đạt được và hỏi `m` có thuộc tập không. Đó là DP.

## 5. Cách 1 — Brute Force: thử mọi tập con

**Ý tưởng:** duyệt mọi `mask` từ `0` đến `2^n - 1`, tính tổng tập con tương ứng; nếu `tổng * 2 == tổng toàn bộ` thì chia được.

In [8]:
def can_partition_brute(nums):
    n = len(nums)
    total = sum(nums)
    for mask in range(1 << n):
        s = 0
        for i in range(n):
            if (mask >> i) & 1:
                s += nums[i]
        if s * 2 == total:
            return True
    return False


for nums, expected in examples:
    got = can_partition_brute(nums)
    print(nums, "->", got)
    assert got == expected

[1, 5, 11, 5] -> True
[1, 2, 3, 5] -> False


**Độ phức tạp:** `2^n` tập con, mỗi lần tính tổng `O(n)` ⇒ Time `O(n * 2^n)`, Space `O(1)`.

**Nút thắt:** `n = 200` thì `2^200` là con số vô vọng; ngay `n = 25` đã `3 * 10^7` tập con × 25 bước. Đo thử với `n` nhỏ để thấy thời gian tăng gấp đôi mỗi lần thêm một phần tử:

In [9]:
for n in [10, 14, 16, 18]:
    arr = [3] * (n - 1) + [1]                         # không chia được -> phải duyệt hết
    t = time.perf_counter()
    r = can_partition_brute(arr)
    dt = time.perf_counter() - t
    print(f"n={n:>2}: kết quả {r}, mất {dt * 1000:9.1f} ms")

n=10: kết quả False, mất       0.8 ms
n=14: kết quả False, mất      18.1 ms
n=16: kết quả False, mất      49.4 ms


n=18: kết quả False, mất     202.4 ms


## 6. Ý tưởng thuật toán / bước đột phá

Hai quan sát:

1. **Quy về một nửa:** nếu tổng `S` lẻ thì `false`. Nếu chẵn, hai tập bằng nhau ⇔ có **một** tập con có tổng `m = S / 2` (phần còn lại tự động có tổng `S - m = m`).
2. **Không cần nhớ cả tập con, chỉ cần nhớ "tổng nào đạt được".** Có nhiều tập con cùng tổng, nhưng ta chỉ quan tâm tổng `j` có đạt được hay không, với `0 <= j <= m` — tối đa `m + 1` giá trị thay vì `2^n` tập.

**Định nghĩa DP:** `f[i][j]` = "có thể chọn một số phần tử trong `i` phần tử đầu để tổng đúng bằng `j`". Với phần tử thứ `i` có giá trị `x`:

- **không chọn** `x`: `f[i][j] = f[i-1][j]`;
- **chọn** `x` (nếu `j >= x`): `f[i][j] = f[i-1][j-x]`.

`f[i][j] = f[i-1][j] or (j >= x and f[i-1][j-x])`. Nền: `f[0][0] = True` (không chọn gì, tổng `0`), `f[0][j>0] = False`. Đáp án `f[n][m]`.

Đây là bài **0-1 knapsack** (mỗi vật dùng tối đa một lần), dạng "có đầy đúng sức chứa `m` không".

## 7. Các cách tiếp theo

### Cách 2 — Đệ quy có nhớ (top-down)

**Giải thích từng đoạn:**

1. Loại trước: tổng lẻ ⇒ `False`. `m = tổng // 2`.
2. Hàm `dfs(i, rest)`: "dùng các phần tử từ chỉ số `i` trở đi, còn cần tạo tổng `rest`, có được không?".
3. Điều kiện dừng: `rest == 0` ⇒ `True` (không cần thêm gì). `i == n` hoặc `rest < 0` ⇒ `False`.
4. Chọn hoặc bỏ: `dfs(i + 1, rest - nums[i]) or dfs(i + 1, rest)`.
5. `@lru_cache` lưu kết quả theo cặp `(i, rest)`: tối đa `n * m` trạng thái.

In [10]:
def can_partition_memo(nums):
    total = sum(nums)
    if total % 2:
        return False
    m = total // 2
    n = len(nums)

    @lru_cache(maxsize=None)
    def dfs(i, rest):
        if rest == 0:
            return True
        if i == n or rest < 0:
            return False
        return dfs(i + 1, rest - nums[i]) or dfs(i + 1, rest)

    return dfs(0, m)


for nums, expected in examples:
    got = can_partition_memo(nums)
    print(nums, "->", got)
    assert got == expected

[1, 5, 11, 5] -> True
[1, 2, 3, 5] -> False


In [11]:
# Code nộp LeetCode — Cách 2 (O(n*m) time, O(n*m) space cho cache + stack đệ quy sâu n)
from functools import lru_cache


class SolutionMemo:
    def canPartition(self, nums: List[int]) -> bool:
        total = sum(nums)
        if total % 2:
            return False
        m = total // 2
        n = len(nums)

        @lru_cache(maxsize=None)
        def dfs(i: int, rest: int) -> bool:
            if rest == 0:
                return True
            if i == n or rest < 0:
                return False
            return dfs(i + 1, rest - nums[i]) or dfs(i + 1, rest)

        return dfs(0, m)


print([SolutionMemo().canPartition(nums) for nums, _ in examples])

[True, False]


**Độ phức tạp Cách 2:** có tối đa `n * (m + 1)` trạng thái `(i, rest)`, mỗi trạng thái `O(1)` ⇒ Time `O(n * m)`, Space `O(n * m)` (cache) và độ sâu đệ quy `O(n)` (tối đa 200, an toàn với giới hạn 1000).

### Cách 3 — Bảng DP 2 chiều (bottom-up)

**Giải thích từng đoạn:**

1. `f = [[False] * (m + 1) for _ in range(n + 1)]`, `f[0][0] = True`.
2. Với `i` từ `1` đến `n`, `x = nums[i-1]`, với `j` từ `0` đến `m`: `f[i][j] = f[i-1][j] or (j >= x and f[i-1][j-x])`.
3. Trả về `f[n][m]`.

In [12]:
def can_partition_2d(nums):
    total = sum(nums)
    if total % 2:
        return False
    m = total // 2
    n = len(nums)
    f = [[False] * (m + 1) for _ in range(n + 1)]
    f[0][0] = True
    for i in range(1, n + 1):
        x = nums[i - 1]
        for j in range(m + 1):
            f[i][j] = f[i - 1][j] or (j >= x and f[i - 1][j - x])
    return f[n][m]


for nums, expected in examples:
    got = can_partition_2d(nums)
    print(nums, "->", got)
    assert got == expected

[1, 5, 11, 5] -> True
[1, 2, 3, 5] -> False


In [13]:
# Code nộp LeetCode — Cách 3 (O(n*m) time, O(n*m) space)
class Solution2D:
    def canPartition(self, nums: List[int]) -> bool:
        total = sum(nums)
        if total % 2:
            return False
        m = total // 2
        n = len(nums)
        f = [[False] * (m + 1) for _ in range(n + 1)]
        f[0][0] = True
        for i in range(1, n + 1):
            x = nums[i - 1]
            for j in range(m + 1):
                f[i][j] = f[i - 1][j] or (j >= x and f[i - 1][j - x])
        return f[n][m]


print([Solution2D().canPartition(nums) for nums, _ in examples])

[True, False]


**Độ phức tạp Cách 3:** Time `O(n * m)`, Space `O(n * m)` — với `n = 200`, `m = 10000` là `2 * 10^6` ô, nhưng dùng nhiều RAM.

### Cách 4 — DP 1 chiều (duyệt ngược `j`)

**Nhận xét:** hàng `i` chỉ dùng hàng `i - 1`. Ta thu gọn về **một mảng** `dp[j]` (= "tổng `j` có đạt được"). Cập nhật tại chỗ: `dp[j] |= dp[j - x]`.

**Vì sao phải duyệt `j` từ `m` xuống `x` (ngược)?** `dp[j - x]` phải là giá trị của **hàng trước** (chưa dùng `x`). Nếu duyệt xuôi, `dp[j - x]` có thể đã được cập nhật bằng chính `x` ở bước này ⇒ dùng `x` **hai lần** (thành bài knapsack **không giới hạn** số lần). Duyệt ngược đảm bảo mọi `dp[j - x]` với `j - x < j` còn là giá trị cũ.

**Giải thích từng đoạn:**

1. `dp = [False] * (m + 1)`, `dp[0] = True`.
2. Với mỗi `x` trong `nums`: với `j` từ `m` xuống `x`: `dp[j] = dp[j] or dp[j - x]`.
3. Trả về `dp[m]`.

In [14]:
def can_partition_1d(nums):
    total = sum(nums)
    if total % 2:
        return False
    m = total // 2
    dp = [False] * (m + 1)
    dp[0] = True
    for x in nums:
        for j in range(m, x - 1, -1):               # duyệt NGƯỢC
            dp[j] = dp[j] or dp[j - x]
    return dp[m]


for nums, expected in examples:
    got = can_partition_1d(nums)
    print(nums, "->", got)
    assert got == expected

[1, 5, 11, 5] -> True
[1, 2, 3, 5] -> False


In [15]:
# Code nộp LeetCode — Cách 4 (O(n*m) time, O(m) space)
class Solution1D:
    def canPartition(self, nums: List[int]) -> bool:
        total = sum(nums)
        if total % 2:
            return False
        m = total // 2
        dp = [False] * (m + 1)
        dp[0] = True
        for x in nums:
            for j in range(m, x - 1, -1):
                dp[j] = dp[j] or dp[j - x]
        return dp[m]


print([Solution1D().canPartition(nums) for nums, _ in examples])

[True, False]


**Độ phức tạp Cách 4:** Time `O(n * m)`, Space `O(m)`.

### Cách 5 — Bitset bằng số nguyên Python

**Ý tưởng:** thay mảng `dp` bằng **một số nguyên** `bits`, bit `j` bật ⇔ tổng `j` đạt được. "Với mọi `j`: `dp[j] |= dp[j - x]`" chính là `bits |= bits << x` (dịch cả dãy bit lên `x` vị trí, rồi hợp với dãy cũ). Python thực hiện thao tác bit trên số nguyên lớn bằng C, xử lý hàng chục bit mỗi lần ⇒ nhanh hơn vòng lặp Python rất nhiều.

**Giải thích từng đoạn:**

1. `bits = 1` (chỉ bit 0 bật: tổng 0).
2. Với mỗi `x`: `bits |= bits << x`. Nếu muốn nhanh hơn có thể che bớt bit `> m`: `bits &= (1 << (m + 1)) - 1`.
3. Kết quả `(bits >> m) & 1 == 1`.

Ở đây không cần lo "dùng một phần tử hai lần": phép `bits << x` luôn tính từ giá trị `bits` **trước** khi gán, tương đương duyệt ngược.

In [16]:
def can_partition_bitset(nums):
    total = sum(nums)
    if total % 2:
        return False
    m = total // 2
    mask = (1 << (m + 1)) - 1                        # giữ lại các bit 0..m
    bits = 1
    for x in nums:
        bits |= bits << x
        bits &= mask
    return (bits >> m) & 1 == 1


for nums, expected in examples:
    got = can_partition_bitset(nums)
    print(nums, "->", got)
    assert got == expected

[1, 5, 11, 5] -> True
[1, 2, 3, 5] -> False


In [17]:
# Code nộp LeetCode — Cách 5 (O(n*m/w) time với w = độ rộng từ máy, O(m/w) space)
class SolutionBitset:
    def canPartition(self, nums: List[int]) -> bool:
        total = sum(nums)
        if total % 2:
            return False
        m = total // 2
        mask = (1 << (m + 1)) - 1
        bits = 1
        for x in nums:
            bits = (bits | (bits << x)) & mask
        return (bits >> m) & 1 == 1


print([SolutionBitset().canPartition(nums) for nums, _ in examples])

[True, False]


**Độ phức tạp Cách 5:** mỗi phần tử một phép dịch trên số có `m` bit (`~m/64` từ máy) ⇒ Time `O(n * m / 64)` về mặt thực tế, Space `O(m / 8)` byte. Trong Python đây thường là cách **nhanh nhất**.

## 8. TRACE từng bước

Trace Cách 4 (DP 1 chiều, duyệt ngược) với `nums = [1, 5, 11, 5]`, `m = 11`. Dưới đây là tập các `j` mà `dp[j] = True` sau mỗi phần tử:

| sau khi xử lý | các tổng đạt được `{j : dp[j]}` (chỉ `j <= 11`) |
|---|---|
| ban đầu | `{0}` |
| `x = 1` | `{0, 1}` |
| `x = 5` | `{0, 1, 5, 6}` |
| `x = 11` | `{0, 1, 5, 6, 11}` |
| `x = 5` | `{0, 1, 5, 6, 10, 11}` |

`dp[11] = True` ⇒ `true`. Hàm `trace_1d` in từng bước nhỏ, kể cả từng `j` bị cập nhật; `trace_bitset` in dạng bit.

In [18]:
def trace_1d(nums):
    total = sum(nums)
    print("nums =", nums, "| tổng =", total)
    if total % 2:
        print("tổng lẻ -> False")
        return False
    m = total // 2
    dp = [False] * (m + 1)
    dp[0] = True
    print("m =", m, "| ban đầu:", [j for j in range(m + 1) if dp[j]])
    for x in nums:
        changed = []
        for j in range(m, x - 1, -1):
            if not dp[j] and dp[j - x]:
                dp[j] = True
                changed.append(f"dp[{j}] <- dp[{j - x}]")
        print(f"x={x:>3}: thêm {changed if changed else 'không có gì'} -> {[j for j in range(m + 1) if dp[j]]}")
    print("dp[m] =", dp[m])
    return dp[m]


trace_1d([1, 5, 11, 5])

nums = [1, 5, 11, 5] | tổng = 22
m = 11 | ban đầu: [0]
x=  1: thêm ['dp[1] <- dp[0]'] -> [0, 1]
x=  5: thêm ['dp[6] <- dp[1]', 'dp[5] <- dp[0]'] -> [0, 1, 5, 6]
x= 11: thêm ['dp[11] <- dp[0]'] -> [0, 1, 5, 6, 11]
x=  5: thêm ['dp[10] <- dp[5]'] -> [0, 1, 5, 6, 10, 11]
dp[m] = True


True

In [19]:
trace_1d([1, 2, 5])           # tổng 8, m = 4: không có tổng 4
print()
trace_1d([1, 2, 3, 5])        # tổng lẻ

nums = [1, 2, 5] | tổng = 8
m = 4 | ban đầu: [0]
x=  1: thêm ['dp[1] <- dp[0]'] -> [0, 1]
x=  2: thêm ['dp[3] <- dp[1]', 'dp[2] <- dp[0]'] -> [0, 1, 2, 3]
x=  5: thêm không có gì -> [0, 1, 2, 3]
dp[m] = False

nums = [1, 2, 3, 5] | tổng = 11
tổng lẻ -> False


False

In [20]:
def trace_bitset(nums):
    total = sum(nums)
    m = total // 2
    width = m + 1
    bits = 1
    print("nums =", nums, "| m =", m)
    print(f"{'ban đầu':>10}: {bits:0{width}b}")
    for x in nums:
        shifted = bits << x
        bits = (bits | shifted) & ((1 << width) - 1)
        print(f"{'x = ' + str(x):>10}: {bits:0{width}b}   (bit thấp nhất bên phải = tổng 0)")
    print("bit m bật?", (bits >> m) & 1 == 1)


trace_bitset([1, 5, 11, 5])

nums = [1, 5, 11, 5] | m = 11
   ban đầu: 000000000001
     x = 1: 000000000011   (bit thấp nhất bên phải = tổng 0)
     x = 5: 000001100011   (bit thấp nhất bên phải = tổng 0)
    x = 11: 100001100011   (bit thấp nhất bên phải = tổng 0)
     x = 5: 110001100011   (bit thấp nhất bên phải = tổng 0)
bit m bật? True


## 9. Code nộp LeetCode

Lời giải chính là **Cách 4** (DP 1 chiều): rõ ràng, `O(n * m)` thời gian, `O(m)` bộ nhớ. Cách 5 (bitset) là phương án nhanh hơn nếu cần.

In [21]:
class Solution:
    def canPartition(self, nums: List[int]) -> bool:
        total = sum(nums)
        if total % 2:                                  # tổng lẻ: không thể chia đều
            return False
        m = total // 2
        dp = [False] * (m + 1)                         # dp[j]: có chọn được tập con tổng j không
        dp[0] = True                                   # tập rỗng có tổng 0
        for x in nums:
            for j in range(m, x - 1, -1):              # duyệt ngược để mỗi x chỉ dùng một lần
                dp[j] = dp[j] or dp[j - x]
        return dp[m]


for nums, expected in examples:
    got = Solution().canPartition(nums)
    print(nums, "->", got)
    assert got == expected

[1, 5, 11, 5] -> True
[1, 2, 3, 5] -> False


## 10. Vì sao đúng

1. **Quy về tổng một nửa:** hai tập con rời nhau phủ hết mảng có tổng bằng nhau ⇔ mỗi tập tổng `S/2`; ngược lại nếu có tập con tổng `S/2` thì phần bù có tổng `S - S/2 = S/2`. `S` lẻ thì không có tổng nguyên `S/2` nên `false`.
2. **Ý nghĩa `dp[j]`:** sau khi xử lý các phần tử `nums[0..i-1]`, `dp[j]` đúng khi và chỉ khi tồn tại tập con của chúng có tổng `j`. Quy nạp: khởi đầu chỉ `dp[0]`. Khi thêm `x`, tập con mới hoặc không chứa `x` (`dp[j]` cũ) hoặc chứa `x` (tập con trước có tổng `j - x`, tức `dp[j - x]` cũ).
3. **Vì sao ngược:** để vế phải `dp[j - x]` là giá trị **cũ**, ta cập nhật `j` từ lớn xuống nhỏ — `dp[j - x]` ở chỉ số nhỏ hơn chưa bị chạm trong lượt này.
4. **Cận `j >= x`:** với `j < x` không thể chọn `x` nên không đổi.
5. **Đủ `m`:** chỉ cần xét `j <= m` vì mọi phần tử dương, tổng vượt `m` không bao giờ quay lại `m`.

## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi tập con | `O(n * 2^n)` | `O(1)` |
| 2. Đệ quy có nhớ | `O(n * m)` | `O(n * m)` |
| 3. Bảng DP 2 chiều | `O(n * m)` | `O(n * m)` |
| 4. DP 1 chiều | **`O(n * m)`** | **`O(m)`** |
| 5. Bitset | `O(n * m / w)` | `O(m / w)` |

Với `n = 200`, `m <= 10000`: `n * m = 2 * 10^6` — rất nhẹ.

## 12. Bộ test

Oracle độc lập: `can_partition_brute` thử mọi tập con bằng bitmask (mục 5). Ta dùng nó cho `n <= 14`; với input lớn đối chiếu giữa các cách DP và bitset, cộng thêm kiểm tra bằng **tính chất** (xây mảng chắc chắn chia được/chắc chắn không).

In [22]:
SOLUTIONS = [Solution(), SolutionMemo(), Solution2D(), Solution1D(), SolutionBitset()]

# 12.1 Ví dụ của đề + edge case
cases = [
    ([1, 5, 11, 5], True),
    ([1, 2, 3, 5], False),
    ([1], False),                              # một phần tử: tổng lẻ
    ([2], False),                              # một phần tử chẵn nhưng không chia được (tập kia rỗng, tổng 0 != 2)
    ([1, 1], True),
    ([100, 100], True),
    ([1, 2, 5], False),                        # tổng chẵn nhưng không có tổng 4
    ([3, 3, 3, 4, 5], True),                   # tổng 18, nửa là 9: [4, 5] và [3, 3, 3]
    ([1, 2, 3, 4, 5, 6, 7], True),             # tổng 28 -> 14
    ([100] * 3, False),                        # tổng 300 chẵn, nhưng không chia được (150 không tạo được)
    ([1] * 200, True),                         # 200 số 1
    ([1] * 199 + [2], False),                  # tổng 201 lẻ
    ([99, 1], False),                          # tổng 100 chẵn, m = 50
]
for nums, expected in cases:
    if len(nums) <= 16:
        assert can_partition_brute(nums) == expected
    for sol in SOLUTIONS:
        got = sol.canPartition(nums)
        assert got == expected, (type(sol).__name__, nums[:10], got, expected)
print("OK:", len(cases), "test x", len(SOLUTIONS), "lớp")

OK: 13 test x 5 lớp


In [23]:
# 12.2 Vét cạn: mọi mảng độ dài <= 6 với giá trị 1..6 (so với oracle bitmask)
from itertools import product

count = 0
for n in range(1, 7):
    for arr in product(range(1, 7), repeat=n):
        nums = list(arr)
        expected = can_partition_brute(nums)
        for sol in SOLUTIONS:
            assert sol.canPartition(nums) == expected, (type(sol).__name__, nums)
        count += 1
print("OK: đối chiếu", count, "mảng với oracle bitmask")

OK: đối chiếu 55986 mảng với oracle bitmask


In [24]:
# 12.3 Ngẫu nhiên có seed: n <= 14, giá trị 1..100, đối chiếu với oracle
random.seed(416)
yes = no = 0
for _ in range(1500):
    n = random.randint(1, 14)
    hi = random.choice([3, 10, 30, 100])
    nums = [random.randint(1, hi) for _ in range(n)]
    expected = can_partition_brute(nums)
    yes += expected
    no += not expected
    for sol in SOLUTIONS:
        assert sol.canPartition(nums) == expected, (type(sol).__name__, nums)
print(f"OK: 1500 mảng ngẫu nhiên (True: {yes}, False: {no})")
assert yes > 100 and no > 100

OK: 1500 mảng ngẫu nhiên (True: 504, False: 996)


In [25]:
# 12.4 Input lớn: n = 200, giá trị tới 100 — các cách thống nhất; kèm kiểm tra bằng cấu trúc
random.seed(4161)
for trial in range(12):
    n = 200
    hi = random.choice([2, 5, 20, 100])
    nums = [random.randint(1, hi) for _ in range(n)]
    results = {sol.canPartition(nums) for sol in SOLUTIONS if not isinstance(sol, SolutionMemo)}
    assert len(results) == 1, results

# mảng chắc chắn chia được: hai nửa có cùng tổng (xây từ hai danh sách bằng tổng)
random.seed(4162)
for _ in range(10):
    a = [random.randint(1, 100) for _ in range(random.randint(1, 99))]
    b = [random.randint(1, 100) for _ in range(random.randint(1, 99))]
    diff = sum(a) - sum(b)
    # chỉnh để tổng bằng nhau bằng cách thêm phần tử vào bên nhỏ hơn (nếu chênh <= 100)
    if 0 < abs(diff) <= 100:
        (b if diff > 0 else a).append(abs(diff))
    if sum(a) == sum(b) and len(a) + len(b) <= 200:
        nums = a + b
        random.shuffle(nums)
        assert all(sol.canPartition(nums) for sol in SOLUTIONS)
# mảng chắc chắn KHÔNG chia được: toàn số chẵn trừ đúng một số lẻ => tổng lẻ
nums = [2 * random.randint(1, 50) for _ in range(199)] + [1]
assert not any(sol.canPartition(nums) for sol in SOLUTIONS)
# tổng chẵn nhưng một phần tử quá lớn (> nửa tổng)
nums = [100] + [1] * 50
assert sum(nums) % 2 == 0 and not any(sol.canPartition(nums) for sol in SOLUTIONS)
print("OK: input lớn + kiểm tra cấu trúc")

OK: input lớn + kiểm tra cấu trúc


In [26]:
# 12.5 Hiệu năng trường hợp xấu: n = 200, giá trị 100 (m xấp xỉ 10000), đáp án False
nums = [100] * 199 + [98]                         # tổng 19998 chẵn, m = 9999 không đạt được (chỉ có bội của 100, hoặc bội của 100 cộng 98)
assert sum(nums) % 2 == 0
for name, fn in [("1D", Solution1D().canPartition), ("Bitset", SolutionBitset().canPartition), ("2D", Solution2D().canPartition)]:
    t = time.perf_counter()
    r = fn(nums)
    dt = time.perf_counter() - t
    print(f"{name:>7}: kết quả {r}, mất {dt * 1000:8.1f} ms")
    assert dt < 10

     1D: kết quả False, mất     80.0 ms
 Bitset: kết quả False, mất      0.1 ms


     2D: kết quả False, mất    164.3 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên kiểm tra tổng lẻ

In [27]:
def wrong_no_parity(nums):
    m = sum(nums) // 2                              # SAI: nếu tổng lẻ, m bị làm tròn xuống
    dp = [False] * (m + 1)
    dp[0] = True
    for x in nums:
        for j in range(m, x - 1, -1):
            dp[j] = dp[j] or dp[j - x]
    return dp[m]


print("sai:", wrong_no_parity([1, 2, 3, 5]), "| đúng:", Solution().canPartition([1, 2, 3, 5]))
bad = [nums for nums in ([1, 2, 3, 5], [1, 1, 1], [2, 3, 4]) if wrong_no_parity(nums) != Solution().canPartition(nums)]
print("input bị sai:", bad)
assert bad

sai: True | đúng: False
input bị sai: [[1, 2, 3, 5], [1, 1, 1], [2, 3, 4]]


**Vì sao sai:** `[1,2,3,5]` có tổng `11`, `11 // 2 = 5`, và tập con `{2, 3}` hoặc `{5}` có tổng `5` ⇒ trả `True`, nhưng phần còn lại có tổng `6 != 5`. **Cách sửa:** `if total % 2: return False` trước khi tính `m`.

### Lỗi 2: duyệt DP 1 chiều theo chiều xuôi

In [28]:
def wrong_forward(nums):
    total = sum(nums)
    if total % 2:
        return False
    m = total // 2
    dp = [False] * (m + 1)
    dp[0] = True
    for x in nums:
        for j in range(x, m + 1):                   # SAI: duyệt xuôi -> dùng x nhiều lần
            dp[j] = dp[j] or dp[j - x]
    return dp[m]


print("sai:", wrong_forward([3, 5, 8]), "| đúng:", Solution().canPartition([3, 5, 8]))
print("sai:", wrong_forward([2, 14]), "| đúng:", Solution().canPartition([2, 14]))
assert wrong_forward([2, 14]) is True and Solution().canPartition([2, 14]) is False

sai: True | đúng: True
sai: True | đúng: False


**Vì sao sai:** `[2, 14]` có tổng `16`, `m = 8`. Duyệt xuôi với `x = 2`: `dp[2], dp[4], dp[6], dp[8]` lần lượt thành `True` vì `dp[j - 2]` vừa được cập nhật — tức dùng phần tử `2` **bốn lần**. Thực tế chỉ có một số `2`. **Cách sửa:** `range(m, x - 1, -1)`.

### Lỗi 3: khởi tạo `dp[0]` sai (hoặc quên)

In [29]:
def wrong_init(nums):
    total = sum(nums)
    if total % 2:
        return False
    m = total // 2
    dp = [False] * (m + 1)                          # SAI: quên dp[0] = True
    for x in nums:
        for j in range(m, x - 1, -1):
            dp[j] = dp[j] or dp[j - x]
    return dp[m]


print("sai:", wrong_init([1, 5, 11, 5]), "| đúng:", Solution().canPartition([1, 5, 11, 5]))
assert wrong_init([1, 5, 11, 5]) is False

sai: False | đúng: True


**Vì sao sai:** `dp[0] = True` là "hạt giống": tổng `0` đạt được bằng tập rỗng. Thiếu nó, không có tổng nào được sinh ra. **Cách sửa:** `dp[0] = True`.

### Lỗi 4: bảng 2 chiều tạo bằng phép nhân danh sách

In [30]:
def wrong_shared_rows(nums):
    total = sum(nums)
    if total % 2:
        return False
    m = total // 2
    n = len(nums)
    f = [[False] * (m + 1)] * (n + 1)               # SAI: mọi hàng là CÙNG một list
    f[0][0] = True
    for i in range(1, n + 1):
        x = nums[i - 1]
        for j in range(m + 1):
            f[i][j] = f[i - 1][j] or (j >= x and f[i - 1][j - x])
    return f[n][m]


bad = [nums for nums in ([1, 2, 5], [2, 14], [3, 3, 3, 4, 5]) if wrong_shared_rows(nums) != Solution().canPartition(nums)]
print("input sai:", bad)
assert bad

input sai: [[1, 2, 5], [2, 14]]


**Vì sao sai:** các hàng trùng đối tượng nên `f[i - 1][j - x]` đã bị ghi đè bằng giá trị của hàng `i` ⇒ giống lỗi duyệt xuôi, dùng phần tử nhiều lần. **Cách sửa:** `[[False] * (m + 1) for _ in range(n + 1)]`.

### Lỗi 5: đệ quy không nhớ

Cách 2 không có `@lru_cache` chạy `O(2^n)` (giống vét cạn). Đo thử:

In [31]:
def no_cache(nums, m):
    n = len(nums)

    def dfs(i, rest):
        if rest == 0:
            return True
        if i == n or rest < 0:
            return False
        return dfs(i + 1, rest - nums[i]) or dfs(i + 1, rest)

    return dfs(0, m)


arr = [3] * 20 + [2]                                # tổng 62, m = 31: không đạt được -> duyệt rất nhiều nhánh
m = sum(arr) // 2
t = time.perf_counter(); r1 = no_cache(arr, m); d1 = time.perf_counter() - t
t = time.perf_counter(); r2 = Solution().canPartition(arr); d2 = time.perf_counter() - t
print(f"không cache: {r1}, {d1 * 1000:.1f} ms | DP: {r2}, {d2 * 1000:.3f} ms")
assert r1 == r2

không cache: False, 184.4 ms | DP: False, 0.062 ms


**Vì sao chậm:** cùng một trạng thái `(i, rest)` bị tính đi tính lại theo cấp số nhân. **Cách sửa:** `@lru_cache` hoặc DP bottom-up.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu nhận dạng:** "chia mảng thành hai phần có tổng bằng nhau / chọn vài phần tử để tổng đúng bằng `T`" ⇒ **0-1 knapsack dạng "đạt đúng tổng"** (subset sum). Cùng họ: *Target Sum* (494), *Last Stone Weight II* (1049), *Ones and Zeroes* (474), *Coin Change* (322, knapsack **không giới hạn**).
- **Quy tắc duyệt:** knapsack 0-1 một chiều ⇒ `j` giảm dần; knapsack không giới hạn (mỗi vật dùng nhiều lần) ⇒ `j` tăng dần.
- **Cắt tỉa nhanh:** tổng lẻ ⇒ `false`; có phần tử `> m` ⇒ `false`; `n == 1` ⇒ `false`.
- **Bitset `bits |= bits << x`** là đoạn code ngắn nhất cho subset sum, và nhanh trong Python.
- Kiểm chứng bằng bitmask trên `n <= 14`.

## 15. Biến thể

1. **Dựng lại một cách chia:** lưu thêm "nguồn" `prev[j]` hoặc dùng bảng 2 chiều rồi truy vết ngược.
2. **Chia thành `k` tập có tổng bằng nhau** (LeetCode 698): cần backtracking/bitmask DP.
3. **Hiệu hai tập nhỏ nhất** (LeetCode 1049): tìm tổng đạt được gần `S/2` nhất — `dp` giống hệt, chỉ khác đáp án.
4. **Đếm số cách chia** (thay `or` bằng `+`): `dp[j] += dp[j - x]`.
5. **Có số âm:** cần dịch chuyển hoặc dùng `set`/`dict` các tổng.

In [32]:
def count_subsets_with_sum(nums, target):
    """Đếm số tập con (theo chỉ số) có tổng = target — knapsack đếm cách."""
    dp = [0] * (target + 1)
    dp[0] = 1
    for x in nums:
        for j in range(target, x - 1, -1):
            dp[j] += dp[j - x]
    return dp[target]


def count_brute(nums, target):
    n = len(nums)
    return sum(1 for mask in range(1 << n) if sum(nums[i] for i in range(n) if (mask >> i) & 1) == target)


print(count_subsets_with_sum([1, 5, 11, 5], 11), count_subsets_with_sum([1, 1, 1, 1], 2))   # 2 6
assert count_subsets_with_sum([1, 5, 11, 5], 11) == count_brute([1, 5, 11, 5], 11) == 2


def min_difference_partition(nums):
    """Biến thể (giống LeetCode 1049): hiệu nhỏ nhất giữa tổng hai tập."""
    total = sum(nums)
    bits = 1
    for x in nums:
        bits |= bits << x
    best = total
    for j in range(total // 2 + 1):
        if (bits >> j) & 1:
            best = min(best, total - 2 * j)
    return best


print(min_difference_partition([1, 5, 11, 5]), min_difference_partition([1, 2, 3, 5]), min_difference_partition([3, 9, 7, 3]))
assert min_difference_partition([1, 5, 11, 5]) == 0 and min_difference_partition([1, 2, 3, 5]) == 1

2 6
0 1 2


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi tập con | bitmask `2^n` | `O(n * 2^n)` | `O(1)` | **oracle** cho `n <= 16` |
| 2. Đệ quy có nhớ | chọn/bỏ + cache `(i, rest)` | `O(n * m)` | `O(n * m)` | dễ nghĩ từ định nghĩa |
| 3. Bảng DP 2 chiều | `f[i][j]` từ `f[i-1]` | `O(n * m)` | `O(n * m)` | dễ truy vết cách chia |
| 4. DP 1 chiều | cập nhật tại chỗ, `j` giảm dần | **`O(n * m)`** | **`O(m)`** | **nộp bài**, phỏng vấn |
| 5. Bitset | `bits \|= bits << x` | `O(n * m / w)` | `O(m / w)` | Python cần tốc độ, code cực ngắn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `divmod(23, 2)`; (b) `list(range(6, 1, -2))`; (c) `bin(1 | (1 << 3))`; (d) `Solution().canPartition([1, 1])`.

**Bài 2 (tính tay).** Với `nums = [2, 3, 5]` (tổng `10`, `m = 5`), liệt kê tập các tổng đạt được sau mỗi phần tử (bắt đầu từ `{0}`) và kết luận `True`/`False`. Làm tiếp với `nums = [2, 4, 7]`.

**Bài 3 (điền TODO).** Hoàn thành `can_partition_todo(nums)` theo Cách 4.

In [33]:
def can_partition_todo(nums):
    # TODO 1: nếu sum(nums) lẻ -> return False; m = sum(nums) // 2
    # TODO 2: dp = [False] * (m + 1); dp[0] = True
    # TODO 3: với mỗi x trong nums: với j từ m xuống x (range(m, x - 1, -1)): dp[j] = dp[j] or dp[j - x]
    # TODO 4: return dp[m]
    return None

# KHÔNG gọi hàm TODO ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `can_partition_bits(nums)` theo Cách 5 (bitset) **không nhìn lại** mục 7. Kiểm tra với các ví dụ của đề và `[1, 2, 5]`.

**Bài 5 (biến thể / mini challenge).** Viết `can_reach_target(nums, target)`: có tập con (mỗi phần tử dùng tối đa một lần) với tổng đúng `target` không? Rồi dùng nó để tự kiểm tra lại: `can_partition(nums) == (sum(nums) % 2 == 0 and can_reach_target(nums, sum(nums) // 2))`. Hãy thử thêm trường hợp `target = 0` (luôn `True`) và `target > sum(nums)` (luôn `False`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `(11, 1)`; `[6, 4, 2]`; `'0b1001'`; `True`. Bài 3: xem `can_partition_1d` ở mục 7.
- **Mức 2 (vừa):** Bài 2: `[2,3,5]`: `{0}` → `{0,2}` → `{0,2,3,5}` → `{0,2,3,5,7,8,10}` ⇒ có `5` ⇒ `True`. `[2,4,7]`: tổng `13` lẻ ⇒ `False` (không cần tính). Bài 4: `bits |= bits << x`, kiểm tra `(bits >> m) & 1`.
- **Mức 3 (sâu):** Bài 5: `bits = 1`; với mỗi `x`: `bits |= bits << x`; trả `(bits >> target) & 1 == 1` (nếu `target > sum(nums)` thì `bits >> target == 0` tự động ra `False`). Với `target = 0`, `bits & 1 == 1` luôn đúng.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [34]:
# Bài 1
assert divmod(23, 2) == (11, 1)
assert list(range(6, 1, -2)) == [6, 4, 2]
assert bin(1 | (1 << 3)) == "0b1001"
assert Solution().canPartition([1, 1]) is True

# Bài 2
reach = {0}
history = []
for x in [2, 3, 5]:
    reach = reach | {s + x for s in reach}
    history.append(sorted(reach))
assert history[-1] == [0, 2, 3, 5, 7, 8, 10] and 5 in reach
assert Solution().canPartition([2, 3, 5]) is True
assert sum([2, 4, 7]) % 2 == 1 and Solution().canPartition([2, 4, 7]) is False


# Bài 3
def can_partition_todo_ans(nums):
    total = sum(nums)
    if total % 2:
        return False
    m = total // 2
    dp = [False] * (m + 1)
    dp[0] = True
    for x in nums:
        for j in range(m, x - 1, -1):
            dp[j] = dp[j] or dp[j - x]
    return dp[m]


for nums, expected in examples:
    assert can_partition_todo_ans(nums) == expected


# Bài 4
def can_partition_bits(nums):
    total = sum(nums)
    if total % 2:
        return False
    bits = 1
    for x in nums:
        bits |= bits << x
    return (bits >> (total // 2)) & 1 == 1


for nums, expected in examples + [([1, 2, 5], False)]:
    assert can_partition_bits(nums) == expected


# Bài 5
def can_reach_target(nums, target):
    bits = 1
    for x in nums:
        bits |= bits << x
    return (bits >> target) & 1 == 1


random.seed(4163)
for _ in range(400):
    nums = [random.randint(1, 20) for _ in range(random.randint(1, 12))]
    s = sum(nums)
    assert Solution().canPartition(nums) == (s % 2 == 0 and can_reach_target(nums, s // 2))
    assert can_reach_target(nums, 0) is True
    assert can_reach_target(nums, s + 1) is False
    t = random.randint(0, s)
    assert can_reach_target(nums, t) == (count_subsets_with_sum(nums, t) > 0)
print("OK: toàn bộ đáp án đúng")

OK: toàn bộ đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template** (0-1 knapsack, DP 1 chiều)

```python
class Solution:
    def canPartition(self, nums: List[int]) -> bool:
        total = sum(nums)
        if total % 2:
            return False
        m = total // 2
        dp = [False] * (m + 1)
        dp[0] = True
        for x in nums:
            for j in range(m, x - 1, -1):
                dp[j] = dp[j] or dp[j - x]
        return dp[m]
```

**Checklist**

- [ ] Kiểm tra **tổng lẻ** trước khi chia đôi?
- [ ] `dp[0] = True` (tập rỗng)?
- [ ] Vòng trong duyệt `j` **từ `m` xuống `x`** (không phải xuôi)?
- [ ] Bảng 2 chiều tạo bằng list comprehension, không phải `[[...]] * n`?
- [ ] Đã thử `[1]`, `[1, 1]`, `[1, 2, 5]` (tổng chẵn nhưng không chia được), `[1] * 200`, `[100] * 200`?
- [ ] Đã đối chiếu với vét cạn bitmask trên mảng ngắn?